In [ ]:
# Install the inference dependencies without replacing Kaggle's PyTorch.
# 安裝推論相依套件，保留 Kaggle 原本的 PyTorch。
%pip -q install "comfy-kitchen==0.2.35" "transformers==4.54.0" "diffusers==0.36.0" "optimum-quanto==0.2.7" "accelerate" "einops" "hf_xet" "huggingface_hub" "safetensors" "psutil" "pillow"

# Clone the pinned WanGP source only when absent.
# 只在不存在時下載固定版本的 WanGP 原始碼。
!test -d /kaggle/working/WanGP/.git || git clone -q https://github.com/deepbeepmeep/WanGP.git /kaggle/working/WanGP

# Check out the original-compatible implementation.
# 使用與原始模型程式相容的 WanGP 版本。
!git -C /kaggle/working/WanGP fetch -q origin 91301f0eda923b6c4b05159e32f1dfd2a960af9d

# Preserve the tested model handler version.
# 保留既有模型載入流程所使用的版本。
!git -C /kaggle/working/WanGP checkout -q 91301f0eda923b6c4b05159e32f1dfd2a960af9d


In [ ]:
# Import the checkpoint downloader.
# 匯入模型權重下載工具。
from pathlib import Path
from huggingface_hub import hf_hub_download

# Store weights outside the Git checkout for repeated use.
# 將模型存於 Git 目錄外，避免重複下載。
CKPT = Path('/kaggle/working/qwen21_ckpts')
# Create the model directory if needed.
# 視需要建立模型目錄。
CKPT.mkdir(parents=True, exist_ok=True)

# List only required image, encoder, VAE, LoRA and tokenizer files.
# 僅列出圖片模型、文字編碼器、VAE、LoRA 與 tokenizer 所需檔案。
MODEL_FILES = [
    ('Comfy-Org/Qwen-Image-2.1', 'diffusion_models/qwen_image_2.1_int8_convrot.safetensors'),
    ('DeepBeepMeep/Ideogram4', 'Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors'),
    ('DeepBeepMeep/Qwen_image_2', 'qwen_image_21/qwen_image_21_vae.safetensors'),
    ('DeepBeepMeep/Qwen_image_2', 'qwen_image_21/vae_config.json'),
    ('DeepBeepMeep/Qwen_image_2', 'qwen_image_21/scheduler_config.json'),
    ('Viggle/Qwen-Image-2.1-viggle-turbo', 'Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors'),
]

# Include the exact Qwen3-VL tokenizer configuration files.
# 補齊 Qwen3-VL 文字編碼器所需的設定檔。
for filename in ('config_legacy.json', 'tokenizer_legacy.json', 'tokenizer_config_legacy.json',
                 'preprocessor_config.json', 'chat_template.jinja', 'merges.txt', 'vocab.json',
                 'added_tokens.json', 'special_tokens_map.json', 'video_preprocessor_config.json'):
    # Reuse the same encoder repository.
    # 沿用相同的編碼器儲存庫。
    MODEL_FILES.append(('DeepBeepMeep/Ideogram4', f'Qwen3-VL-8B-Instruct/{filename}'))

# Reuse existing completed files whenever possible.
# 優先沿用已完成下載的模型檔案。
for repo, filename in MODEL_FILES:
    # Download only missing or incomplete Hugging Face assets.
    # 只補下載尚未完整取得的 Hugging Face 檔案。
    hf_hub_download(repo_id=repo, filename=filename, local_dir=str(CKPT))

# Confirm that the checkpoint set is complete.
# 顯示模型下載完成訊息。
print('Models ready:', CKPT)


In [ ]:
%%writefile /kaggle/working/qwen21_worker.py
# Keep each GPU worker independent to avoid unsupported model partitioning.
# 各 GPU 使用獨立程序，避免不支援的跨 GPU 模型切分。
import os
import sys
import time
import types
import traceback
from pathlib import Path
from multiprocessing.connection import Listener

# Disable optional framework imports before model initialization.
# 初始化前關閉不必要的深度學習框架載入。
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Load the pinned project modules from Kaggle's working directory.
# 從 Kaggle 工作目錄載入固定版本的模組。
sys.path.insert(0, '/kaggle/working/WanGP')

# Stub out unused background removal imports.
# 跳過不需要的背景移除套件匯入。
rembg = types.ModuleType('rembg')

# Fail explicitly if optional background removal is requested.
# 若誤用未啟用的背景移除功能，明確回報錯誤。
def unsupported_rembg(*args, **kwargs):
    # Avoid silently returning invalid images.
    # 避免默默回傳無效圖片。
    raise RuntimeError('Background removal is disabled.')

# Provide the optional module symbols expected by WanGP.
# 提供 WanGP 預期的選用模組名稱。
rembg.remove = rembg.new_session = unsupported_rembg
sys.modules.setdefault('rembg', rembg)

# Import the necessary GPU and image libraries.
# 匯入 GPU 與影像處理套件。
import numpy as np
import torch
from PIL import Image
from mmgp import offload, quant_router
from optimum.quanto.tensor.weights import qbytes
from shared.utils import files_locator
from shared.kernels import int8_backend, quanto_int8_inject
from models.qwen21.qwen21_handler import family_handler, ENCODER

# Read the worker configuration passed by the notebook.
# 讀取 Notebook 傳入的常駐服務設定。
PORT = int(sys.argv[1])
AUTH = bytes.fromhex(sys.argv[2])
CKPT = Path('/kaggle/working/qwen21_ckpts')
DTYPE = torch.float16 if torch.cuda.get_device_properties(0).major < 8 else torch.bfloat16

# Register the INT8 ConvRot loader used by Qwen-Image 2.1.
# 註冊 Qwen-Image 2.1 的 INT8 ConvRot 權重格式。
quant_router.unregister_handler('.fp8_quanto_bridge')
quant_router.register_handler('shared.qtypes.int8_convrot')
files_locator.set_checkpoints_paths([str(CKPT)])

# Prefer the INT8 acceleration available on T4.
# 優先使用 T4 支援的 INT8 運算。
try:
    # Enable the automatic INT8 kernel choice.
    # 啟用自動 INT8 kernel 選擇。
    int8_backend.configure('auto', 1)
except Exception as error:
    # Keep the supported PyTorch fallback available.
    # 保留可相容的 PyTorch 後備路徑。
    print('INT8 fallback:', error, flush=True)

# Reserve a bounded workspace for quantized kernels.
# 限制量化 kernel 使用的工作空間。
int8_backend._SCRATCH_BYTES = 384 * 1024 * 1024

# Keep the large word embedding table in CPU memory.
# 將大型詞嵌入表保留在 CPU 記憶體。
class CPUEmbedding(torch.nn.Module):
    # Retain the original pretrained embedding weights.
    # 保留原本預訓練的 embedding 權重。
    def __init__(self, weight):
        # Initialize its module base.
        # 初始化模組基底。
        super().__init__()
        # Avoid registering the entire table as a GPU parameter.
        # 避免將整張表註冊為 GPU 參數。
        self.table = [weight]

    # Transfer only the requested embedding vectors.
    # 只傳輸目前 token 需要的 embedding 向量。
    def forward(self, input_ids):
        # Keep embedding lookup on CPU.
        # 在 CPU 查詢 embedding。
        return torch.nn.functional.embedding(input_ids.cpu(), self.table[0]).to(input_ids.device)

# Preserve the original model loader to restore it after initialization.
# 保存原始模型載入函式，避免影響後續操作。
original_loader = offload.fast_load_transformers_model

# Supply the GPU-compatible dtype when loading model components.
# 載入模型元件時使用適合 GPU 的資料型別。
def load_with_dtype(*args, **kwargs):
    # Use FP16 on T4 unless the caller overrides it.
    # T4 預設採用 FP16，除非呼叫端另有指定。
    kwargs.setdefault('default_dtype', DTYPE)
    # Delegate the remaining logic to WanGP.
    # 其餘載入工作仍交由 WanGP。
    return original_loader(*args, **kwargs)

# Apply the temporary dtype-compatible loader.
# 暫時套用相容的 dtype 載入函式。
offload.fast_load_transformers_model = load_with_dtype

# Load the model exactly once for this GPU worker.
# 這個 GPU 程序只載入一次模型。
try:
    # Resolve the original Qwen-Image 2.1 model structure.
    # 取得原本的 Qwen-Image 2.1 模型結構。
    model_def = family_handler.query_model_def('qwen_image_21_7B', {})
    # Load the INT8 transformer, text/vision encoder and VAE.
    # 載入 INT8 Transformer、文字／視覺編碼器及 VAE。
    pipeline, parts = family_handler.load_model(
        [str(CKPT / 'diffusion_models/qwen_image_2.1_int8_convrot.safetensors')],
        'qwen_image_21_7B', 'qwen_image_21_7B', model_def,
        text_encoder_filename=str(CKPT / ENCODER / f'{ENCODER}_int8_convrot.safetensors'),
    )
finally:
    # Undo the temporary loader patch, even on failure.
    # 即使失敗也恢復原始載入函式。
    offload.fast_load_transformers_model = original_loader

# Access the models managed by the same pipeline.
# 取得同一個 Pipeline 管理的各個模型。
transformer = parts['transformer']
text_encoder = parts['text_encoder']

# Keep the text embedding table outside the GPU manager.
# 將文字 embedding 表排除於 GPU 管理範圍之外。
text_encoder.embed_tokens = CPUEmbedding(text_encoder.embed_tokens.weight.data)

# Keep model objects in RAM and stream only blocks needed on the GPU.
# 模型物件常駐 RAM，GPU 依需求載入部分權重，避免 T4 OOM。
manager = offload.profile(
    {'transformer': transformer, 'text_encoder': text_encoder,
     'vision_encoder': parts['vision_encoder'], 'vae': parts['vae']},
    profile_no=5, pinnedMemory=False, quantizeTransformer=False,
    extraModelsToQuantize=[],
    budgets={'transformer': 6144, 'text_encoder': 2048,
             'vision_encoder': 1024, 'vae': 2048},
    coTenantsMap={'transformer': '*', 'vae': '*'},
    loras=['transformer'], convertWeightsFloatTo=DTYPE,
    asyncTransfers=True,
)

# Use a memory-efficient attention backend on T4.
# T4 採用較省顯示記憶體的 attention backend。
offload.shared_state['_attention'] = 'sdpa'

# Load the 6-step Turbo adapter once per worker.
# 每個程序只載入一次 6-step Turbo LoRA。
lora_path = CKPT / 'Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors'
loaded = offload.load_loras_into_model(
    transformer, [str(lora_path)], [1.0], activate_all_loras=True,
    pinnedLora=False,
    preprocess_sd=lambda sd: transformer.preprocess_loras('qwen_image_21_7B', sd),
    split_linear_modules_map=getattr(transformer, 'split_linear_modules_map', None),
)

# Ensure that the Turbo weights are actually applied.
# 確認 Turbo 權重確實套用成功。
assert loaded, 'Turbo LoRA failed to load.'
offload.activate_loras(transformer, ['0'], [1.0])

# Save the original text-conditioning implementation.
# 保存原始文字條件編碼函式。
original_encoder = pipeline._encode_conditioning

# Work around INT8 text-encoding precision issues on T4.
# 修正 T4 上 INT8 文字編碼的精度相容性。
def encode_prompt(prompts, images, device, batch_size):
    # Record the original INT8 kernel and autograd function.
    # 保存原本的 INT8 backend 與運算函式。
    backend = int8_backend._backend
    original_forward = qbytes.WeightQBytesLinearFunction.__dict__['forward']
    # Use the accurate fallback during encoding.
    # 編碼期間使用精確的後備運算。
    int8_backend._backend = 'pytorch'
    qbytes.WeightQBytesLinearFunction.forward = staticmethod(
        quanto_int8_inject._default_quanto_qbytes_linear_forward
    )
    try:
        # Encode on the same managed GPU as the rest of this worker.
        # 在此程序的同一張 GPU 上進行編碼，避免跨卡張量衝突。
        contexts = original_encoder(prompts, images, device, batch_size)
    finally:
        # Restore original INT8 behavior for image generation.
        # 還原生成模型使用的 INT8 運算。
        int8_backend._backend = backend
        qbytes.WeightQBytesLinearFunction.forward = original_forward

    # Convert conditioning to the model's supported dtype.
    # 將條件向量轉成模型支援的 dtype。
    converted = []
    for embeddings, mask, slots in contexts:
        # Rescale overflowing values before FP16 conversion.
        # 轉換 FP16 前先避免過大值溢位。
        if DTYPE == torch.float16:
            # Estimate the maximum per-token embedding magnitude.
            # 估算每個 token embedding 的最大數值。
            peak = embeddings.float().abs().amax(dim=-1, keepdim=True)
            # Preserve normal values while limiting extreme magnitudes.
            # 保留正常數值並限制極端 magnitude。
            embeddings = embeddings.float() / (peak / 32768).clamp(min=1)
        # Keep the resulting tensor on its original managed device.
        # 將張量保留在原本由模型管理的裝置。
        converted.append((embeddings.to(DTYPE), mask, slots))
    # Return the complete text-conditioning bundle.
    # 回傳完整文字條件資料。
    return converted

# Install the correction only once for this worker.
# 此程序只安裝一次修正。
pipeline._encode_conditioning = encode_prompt

# Resize supplied images while preserving aspect ratio.
# 保持參考圖片的長寬比並控制 token 數量。
def read_references(paths, target_size):
    # Store the ordered references for editing and composition.
    # 依序保存編輯或合成的參考圖。
    references = []
    # Read every requested source image.
    # 讀取每張指定的參考圖。
    for raw_path in paths:
        # Reject missing references before generating.
        # 生成前檢查參考圖是否存在。
        path = Path(raw_path)
        if not path.is_file():
            # Explain which source path failed.
            # 指出遺失的圖片路徑。
            raise FileNotFoundError(path)
        # Load the requested image once for this job.
        # 每次任務只開啟一次指定圖片。
        with Image.open(path) as source:
            # Preserve alpha only when transparency exists.
            # 僅在有透明資訊時保留 alpha。
            has_alpha = source.mode in ('RGBA', 'LA') or ('transparency' in source.info)
            # Convert to a format accepted by Qwen-Image.
            # 轉換為 Qwen-Image 可接受的格式。
            image = source.convert('RGBA' if has_alpha else 'RGB')
        # Calculate a size aligned to model patch dimensions.
        # 計算符合模型 patch 對齊需求的大小。
        scale = (target_size ** 2 / (image.width * image.height)) ** 0.5
        size = tuple(max(32, round(n * scale / 32) * 32) for n in image.size)
        # Keep all reference images in the prompt's original order.
        # 保留提示詞中參考圖片的原始順序。
        references.append(image.resize(size, Image.Resampling.LANCZOS))
    # Supply the references to the model pipeline.
    # 將參考圖提供給生成 Pipeline。
    return references

# Perform inference without reinitializing the weights.
# 不重新載入權重，直接使用現有 Pipeline 生成圖片。
def generate(config):
    # Read and validate the requested generation mode.
    # 讀取並驗證生成模式。
    task = config['task']
    paths = config['references']
    width, height = int(config['width']), int(config['height'])
    # Validate dimensions and reference counts.
    # 驗證解析度與參考圖數量。
    if task not in ('text', 'edit', 'multi') or width % 32 or height % 32:
        # Reject unsupported tasks and unaligned image dimensions.
        # 拒絕不支援的模式及未對齊的尺寸。
        raise ValueError('Invalid task or width/height (multiples of 32 required).')
    if (task == 'edit' and len(paths) != 1) or (task == 'multi' and len(paths) not in (2, 3)):
        # Require a compatible number of reference images.
        # 要求提供與模式相符的參考圖片數量。
        raise ValueError('edit needs 1 reference; multi needs 2–3 references.')
    if task == 'text' and paths:
        # Avoid silently ignoring accidental image inputs.
        # 避免默默忽略誤填的圖片來源。
        raise ValueError('text mode does not accept references.')

    # Load reference image pixels only for the current job.
    # 只有這次工作需要時才載入參考圖。
    references = read_references(paths, config['reference_size'])
    # Measure inference without replacing or unloading models.
    # 記錄生成耗時，不重建或卸載模型。
    start = time.perf_counter()
    # Disable autograd to reduce memory usage.
    # 停用 autograd 以降低記憶體消耗。
    with torch.inference_mode():
        # Reuse the pinned pipeline and Turbo sampler.
        # 重複使用已載入的 Pipeline 與 Turbo Sampler。
        result = pipeline.generate(
            input_prompt=config['prompt'], n_prompt=None, seed=int(config['seed']),
            width=width, height=height, sampling_steps=6,
            sample_solver='viggle_v02', guide_scale=1.0,
            batch_size=1, VAE_tile_size=(True, 256), loras_slists=None,
            input_ref_images=references or None,
            custom_settings={'qwen21_kv_cache': 'Disabled', 'rgba': 'Disabled'},
        )
    # Stop instead of saving an incomplete output.
    # 生成失敗時停止，避免儲存不完整圖片。
    if result is None:
        # Fail early when model output is empty.
        # 沒有輸出時提早回報錯誤。
        raise RuntimeError('The model returned no image.')

    # Convert the result to a standard HWC image.
    # 將輸出轉換成一般 HWC 圖片。
    array = result[:, 0].permute(1, 2, 0).contiguous().cpu().numpy()
    # Preserve correct pixel ranges for both float and uint8 results.
    # 兼容 float 與 uint8 的像素數值範圍。
    if np.issubdtype(array.dtype, np.floating) and array.max() <= 1.0:
        # Convert normalized pixels to 8-bit values.
        # 將正規化像素轉為 8-bit 值。
        array = array * 255.0
    # Store the generated image in the shared output folder.
    # 將生成圖片儲存於共用輸出資料夾。
    output = Path(config['output'])
    # Ensure the destination directory exists.
    # 確認輸出目錄存在。
    output.parent.mkdir(parents=True, exist_ok=True)
    # Clamp extreme values to the valid PNG range.
    # 將極端數值限制於 PNG 有效範圍。
    Image.fromarray(np.clip(array, 0, 255).astype(np.uint8)).save(output)
    # Return completion time and current VRAM use for this worker.
    # 回傳完成時間與此程序的 GPU 使用量。
    return {'path': str(output), 'seconds': round(time.perf_counter() - start, 1),
            'gpu_memory_gib': round(torch.cuda.memory_allocated() / 1024 ** 3, 2)}

# Start a persistent local service only after the models are ready.
# 模型完成載入後才啟動常駐服務。
with Listener(('127.0.0.1', PORT), authkey=AUTH) as listener:
    # Report worker readiness in its log.
    # 在 Log 中記錄服務已就緒。
    print(f'GPU worker ready on port {PORT}', flush=True)
    # Process jobs indefinitely until this Kaggle session ends.
    # 在這次 Kaggle Session 中持續處理生成任務。
    while True:
        # Accept one request at a time for safe model reuse.
        # 每個 GPU 一次只處理一個請求，安全重用模型。
        with listener.accept() as connection:
            # Receive the generation configuration from the notebook.
            # 接收 Notebook 傳來的生成設定。
            request = connection.recv()
            # Interpret lightweight worker control messages.
            # 判讀輕量的服務控制指令。
            if request['command'] == 'ping':
                # Return readiness without running inference.
                # 不執行推論，直接回報已就緒。
                connection.send({'ok': True, 'message': 'ready'})
                # Continue to the next request.
                # 等待下一個任務。
                continue
            try:
                # Run the image job using cached models.
                # 使用已快取的模型執行圖片生成。
                info = generate(request['config'])
                # Report the generated file and timing.
                # 回傳圖片位置與耗時。
                connection.send({'ok': True, **info})
            except Exception:
                # Report the actual error without terminating model residency.
                # 回報錯誤，但不主動停止模型常駐服務。
                connection.send({'ok': False, 'error': traceback.format_exc()[-3000:]})


In [ ]:
# Start persistent inference workers on the two Kaggle GPUs.
# 在兩張 Kaggle GPU 啟動各自的常駐推論服務。
import os
import subprocess
import sys
import time
from multiprocessing.connection import Client
from pathlib import Path
import torch

# Keep one worker per GPU; turn GPU 1 off if host RAM is insufficient.
# 每張 GPU 一個服務；主機 RAM 不足時可停用 GPU 1。
WORKER_GPUS = [0, 1]

# Verify the required GPUs are visible to Kaggle.
# 確認 Kaggle 已提供所需的 GPU。
assert torch.cuda.device_count() > max(WORKER_GPUS), '請啟用 Kaggle T4 ×2。'

# Store worker processes and the shared session authentication key.
# 保留服務程序及本次 Session 的驗證金鑰。
QWEN21_WORKERS = globals().get('QWEN21_WORKERS', {})
QWEN21_AUTH = globals().get('QWEN21_AUTH', os.urandom(16))

# Use a lightweight local connection to submit generation requests.
# 透過本機輕量連線提交生成請求。
def call_worker(gpu, command, config=None):
    # Connect to the specific GPU's persistent process.
    # 連線至指定 GPU 的常駐程序。
    with Client(('127.0.0.1', 29650 + gpu), authkey=QWEN21_AUTH) as connection:
        # Send the requested action and parameters.
        # 傳送操作與參數。
        connection.send({'command': command, 'config': config})
        # Read the worker's response.
        # 讀取服務回傳結果。
        return connection.recv()

# Launch only missing or terminated GPU workers.
# 只啟動尚未存在或已結束的 GPU 程序。
for gpu in WORKER_GPUS:
    # Reuse an existing live model service.
    # 重複使用仍在運作的模型服務。
    process = QWEN21_WORKERS.get(gpu)
    if process is not None and process.poll() is None:
        # Reuse the same memory-resident weights.
        # 沿用已載入記憶體的權重。
        print(f'GPU {gpu}: 已沿用模型服務。', flush=True)
        continue

    # Make this worker see only its designated physical GPU.
    # 讓此程序只看見指定的實體 GPU。
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = str(gpu)
    environment['USE_TF'] = '0'
    environment['USE_FLAX'] = '0'
    # Capture initialization diagnostics in a separate log file.
    # 將各 GPU 載入紀錄存入獨立 Log。
    log_path = Path(f'/kaggle/working/qwen21_gpu{gpu}.log')
    log_stream = log_path.open('w', encoding='utf-8')
    # Spawn a separate process to avoid unsafe CUDA sharing.
    # 使用獨立程序，避免不安全的跨程序 CUDA 共用。
    process = subprocess.Popen(
        [sys.executable, '-u', '/kaggle/working/qwen21_worker.py',
         str(29650 + gpu), QWEN21_AUTH.hex()],
        cwd='/kaggle/working/WanGP', env=environment,
        stdout=log_stream, stderr=subprocess.STDOUT,
    )
    # Close the parent's log descriptor; the worker retains its own.
    # 關閉父程序的 Log 描述符，子程序仍可持續寫入。
    log_stream.close()
    QWEN21_WORKERS[gpu] = process

    # Wait for the model initialization to complete or fail.
    # 確認模型載入完成或回報失敗。
    deadline = time.monotonic() + 900
    while True:
        # Fail with the log when a worker exits unexpectedly.
        # 子程序異常結束時顯示完整錯誤尾端。
        if process.poll() is not None:
            # Include the relevant diagnostic lines.
            # 提供近期錯誤紀錄以利排查。
            raise RuntimeError(f'GPU {gpu} 初始化失敗:\n' + log_path.read_text(errors='replace')[-5000:])
        try:
            # Check readiness without causing model inference.
            # 只檢查狀態，不執行模型推論。
            if call_worker(gpu, 'ping')['ok']:
                # Report verified worker readiness.
                # 顯示已驗證的服務狀態。
                print(f'GPU {gpu}: 模型載入完成並常駐。', flush=True)
                break
        except (OSError, ConnectionError):
            # The worker is still loading checkpoints.
            # 程序仍在載入模型權重。
            pass
        # Avoid waiting forever if a worker cannot initialize.
        # 避免初始化失敗時無限制等待。
        if time.monotonic() > deadline:
            # Show the last model-loading log for diagnosis.
            # 顯示最後一段載入紀錄供診斷。
            raise TimeoutError(f'GPU {gpu} 初始化逾時:\n' + log_path.read_text(errors='replace')[-5000:])
        # Poll at a modest interval during model loading.
        # 模型載入期間採適當間隔檢查。
        time.sleep(2)


In [ ]:
# Select text generation, single-image editing or multi-reference generation.
# 選擇文生圖、單張編輯或多參考圖合成。
TASK = 'text'                  # 'text', 'edit', 'multi'

# Describe the target image or desired edits.
# 輸入生成內容或想修改的圖片特徵。
PROMPT = 'A cozy bookstore in Taipei on a rainy evening, realistic photography'

# Keep image size within a comfortable T4 range.
# 將尺寸控制在 T4 較容易處理的範圍。
WIDTH, HEIGHT = 640, 640

# Set the starting seed; multiple GPUs use consecutive seeds.
# 設定起始 Seed；多張 GPU 各使用連續 Seed。
SEED = 42

# Choose output filename, without overwriting older results.
# 設定檔名，自動避免覆蓋既有結果。
OUTPUT_NAME = 'qwen21.png'

# Use existing Kaggle paths for reference images (edit: 1; multi: 2–3).
# 參考圖請填 Kaggle 路徑（edit：1 張；multi：2～3 張）。
REFERENCE_PATHS = []

# Limit reference resolution to prevent excessive encoder memory usage.
# 限制參考圖片尺寸，避免編碼器佔用過多記憶體。
REFERENCE_SIZE = 384

# Use both GPUs for two images, or set [0] for one output.
# 兩張 GPU 各生成一張；只要一張可改為 [0]。
GENERATE_GPUS = [0, 1]


In [ ]:
# Submit jobs to the already-loaded GPU model services.
# 將生成任務交給已載入模型的常駐服務。
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import monotonic, sleep
from IPython.display import display, FileLink, Image as NotebookImage

# Validate model services and user settings before generating.
# 生成前驗證模型服務及使用者設定。
assert set(GENERATE_GPUS).issubset(QWEN21_WORKERS), '請先執行「啟動模型服務」。'
assert TASK in ('text', 'edit', 'multi'), '未知的 TASK 設定。'
assert WIDTH % 32 == HEIGHT % 32 == 0 and min(WIDTH, HEIGHT) >= 256, '尺寸必須為 32 的倍數且 >= 256。'

# Create a persistent folder for all generated images.
# 建立保存生成圖片的專用目錄。
OUTPUT_DIR = Path('/kaggle/working/qwen21_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Normalize the filename to a safe PNG basename.
# 將檔名整理成安全的 PNG 名稱。
base = Path(OUTPUT_NAME).name
if not base.lower().endswith('.png'):
    # Always save PNG output.
    # 統一使用 PNG 輸出。
    base += '.png'

# Build a unique output path for each GPU without overwriting files.
# 為每張 GPU 建立不覆蓋舊檔的輸出路徑。
def next_path(gpu):
    # Add a GPU suffix when generating two outputs in parallel.
    # 同時產出兩張時，在檔名加入 GPU 編號。
    name = f'{Path(base).stem}_gpu{gpu}.png' if len(GENERATE_GPUS) > 1 else base
    # Keep a stable candidate in the output folder.
    # 先使用指定資料夾內的基本檔名。
    candidate = OUTPUT_DIR / name
    # Append a numeric suffix if the file already exists.
    # 檔案存在時自動加入數字編號。
    number = 2
    while candidate.exists():
        # Keep every generated image for later comparisons.
        # 保留每次生成結果供後續比較。
        candidate = OUTPUT_DIR / f'{Path(name).stem}_{number:03d}.png'
        number += 1
    # Return this job's destination.
    # 回傳這次任務的輸出位置。
    return candidate

# Prepare per-GPU generation settings from the same prompt.
# 以相同 Prompt 建立各 GPU 的生成設定。
jobs = {}
for gpu in GENERATE_GPUS:
    # Ensure the persistent process is still running.
    # 確認常駐程序仍在執行。
    assert QWEN21_WORKERS[gpu].poll() is None, f'GPU {gpu} 服務已停止，請重跑初始化。'
    # Pass only ordinary data to the independent GPU processes.
    # 僅向獨立 GPU 程序傳送一般設定資料。
    jobs[gpu] = {
        'task': TASK, 'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT,
        'seed': SEED + gpu, 'references': list(REFERENCE_PATHS),
        'reference_size': REFERENCE_SIZE, 'output': str(next_path(gpu)),
    }

# Send both jobs concurrently so both T4s can be used.
# 平行送出兩個工作，讓兩張 T4 同時執行。
with ThreadPoolExecutor(max_workers=len(jobs)) as executor:
    # Submit one job to each selected GPU.
    # 向每張指定 GPU 送出一個任務。
    futures = {gpu: executor.submit(call_worker, gpu, 'generate', config)
               for gpu, config in jobs.items()}
    # Display an initial status below the notebook cell.
    # 在 Notebook Cell 下方顯示起始狀態。
    print('生成中：', ', '.join(f'GPU {gpu}' for gpu in jobs), flush=True)
    # Track elapsed time while requests are running.
    # 在等待生成時持續顯示已耗費時間。
    started = monotonic()
    while not all(future.done() for future in futures.values()):
        # Avoid a completely silent generation cell.
        # 避免生成 Cell 長時間沒有輸出。
        sleep(10)
        # Report progress as elapsed time, without fabricating a percentage.
        # 顯示已耗費時間，不虛構進度百分比。
        print(f'已執行 {monotonic() - started:.0f} 秒', flush=True)

# Preview each successful output and report any failure separately.
# 預覽成功圖片，並個別顯示失敗訊息。
OUTPUTS = []
for gpu, future in futures.items():
    # Read the completed worker response.
    # 讀取各 GPU 的執行結果。
    try:
        # Read the worker response even if another GPU failed.
        # 即使另一張 GPU 失敗，也能獨立讀取本張的結果。
        response = future.result()
    except Exception as error:
        # Report connection loss or a terminated worker clearly.
        # 清楚顯示連線中斷或 GPU 程序結束等錯誤。
        print(f'GPU {gpu} 服務連線失敗：{error}', flush=True)
        continue
    if not response['ok']:
        # Surface the actual stack trace for troubleshooting.
        # 顯示實際錯誤堆疊，方便診斷。
        print(f'GPU {gpu} 發生錯誤：\n{response["error"]}', flush=True)
        continue
    # Keep track of all outputs in this generation round.
    # 紀錄這次生成的所有檔案。
    path = Path(response['path'])
    OUTPUTS.append(path)
    # Show timing, GPU memory, the image and a clickable file link.
    # 顯示耗時、GPU 記憶體、圖片與檔案連結。
    print(f'GPU {gpu}: {response["seconds"]} 秒 | VRAM {response["gpu_memory_gib"]} GiB | {path}', flush=True)
    display(NotebookImage(filename=str(path), width=min(480, WIDTH)), FileLink(str(path)))

# Stop when all GPU jobs failed.
# 當所有 GPU 都失敗時明確回報錯誤。
assert OUTPUTS, '本次沒有成功生成圖片，請查看上方錯誤。'
